# Testes do motor de dosimetria

Cobre `Fracao`, `Pena` e `Faixa` (`dosimetria/`). Rode as células em ordem ("Run All"); qualquer `assert` que falhar interrompe a execução naquele ponto e mostra o traceback.

In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "dosimetria").is_dir():
        sys.path.insert(0, str(candidate))
        break

from dosimetria import Fracao, Pena, Faixa

## Fracao

In [2]:
assert Fracao(1, 3).aplicar(360) == 120

# 1/3 de 365 = 121.66... -> desprezar a fração de dia (art. 11 do CP)
assert Fracao(1, 3).aplicar(365) == 121

assert Fracao(1, 2).aplicar(0) == 0

for numerador, denominador in [(1, 0), (1, -3)]:
    try:
        Fracao(numerador, denominador)
        raise AssertionError(f"deveria ter rejeitado denominador={denominador}")
    except ValueError:
        pass

try:
    Fracao(-1, 3)
    raise AssertionError("deveria ter rejeitado numerador negativo")
except ValueError:
    pass

assert str(Fracao(1, 6)) == "1/6"

print("Fracao: OK")

Fracao: OK


## Pena

In [3]:
# convenção do projeto: 1 ano = 365 dias, 1 mês = 30 dias
assert Pena.de_anos_meses_dias(anos=1).dias == 365
assert Pena.de_anos_meses_dias(meses=1).dias == 30
assert Pena.de_anos_meses_dias(anos=1, meses=2, dias=3).dias == 365 + 60 + 3

try:
    Pena(-1)
    raise AssertionError("deveria ter rejeitado pena negativa")
except ValueError:
    pass

assert Pena(360).mais(Fracao(1, 3)).dias == 480
assert Pena(360).menos(Fracao(1, 6)).dias == 300

assert Pena(100).mais_dias(50).dias == 150
assert Pena(100).menos_dias(50).dias == 50

try:
    Pena(10).menos_dias(20)
    raise AssertionError("deveria ter rejeitado pena negativa")
except ValueError:
    pass

assert Pena.de_anos_meses_dias(anos=1, meses=4, dias=4).como_anos_meses_dias() == (1, 4, 4)

assert Pena(100) < Pena(200)
assert Pena(200) > Pena(100)
assert Pena(100) <= Pena(100)
assert Pena(100) == Pena(100)

assert str(Pena.de_anos_meses_dias(anos=1, meses=4, dias=4)) == "1 ano, 4 meses, 4 dias"
assert str(Pena(0)) == "0 dias"

print("Pena: OK")

Pena: OK


## Faixa

In [4]:
faixa_furto_simples = Faixa(
    minimo=Pena.de_anos_meses_dias(anos=1),
    maximo=Pena.de_anos_meses_dias(anos=4),
    origem="CP.art155",
)

assert faixa_furto_simples.contem(Pena.de_anos_meses_dias(anos=2))
assert faixa_furto_simples.contem(faixa_furto_simples.minimo)
assert faixa_furto_simples.contem(faixa_furto_simples.maximo)
assert not faixa_furto_simples.contem(Pena.de_anos_meses_dias(anos=5))

assert faixa_furto_simples.limitar(Pena.de_anos_meses_dias(meses=6)) == faixa_furto_simples.minimo
assert faixa_furto_simples.limitar(Pena.de_anos_meses_dias(anos=10)) == faixa_furto_simples.maximo

pena_dentro = Pena.de_anos_meses_dias(anos=2)
assert faixa_furto_simples.limitar(pena_dentro) == pena_dentro

try:
    Faixa(minimo=Pena(100), maximo=Pena(50), origem="teste")
    raise AssertionError("deveria ter rejeitado mínimo > máximo")
except ValueError:
    pass

print("Faixa: OK")

Faixa: OK


## Quantum (estratégias de incremento por circunstância)

In [5]:
from dosimetria import EstrategiaQuantum, FracaoDoIntervalo, FracaoDoMinimo

faixa_furto = Faixa(
    minimo=Pena.de_anos_meses_dias(anos=1),
    maximo=Pena.de_anos_meses_dias(anos=4),
    origem="CP.art155",
)

# intervalo = 3 anos = 1095 dias; 1/8 do intervalo
assert FracaoDoIntervalo().incremento_por_circunstancia(faixa_furto).dias == Fracao(1, 8).aplicar(1095)

# 1/6 do mínimo (365 dias)
assert FracaoDoMinimo().incremento_por_circunstancia(faixa_furto).dias == Fracao(1, 6).aplicar(365)

assert isinstance(FracaoDoIntervalo(), EstrategiaQuantum)
assert "1/8" in FracaoDoIntervalo().nome

print("Quantum: OK")

Quantum: OK


## Fase 1 — pena-base (art. 59 do CP)

In [6]:
from dosimetria import CircunstanciaJudicial, Valoracao, calcular_pena_base

TODAS_NEUTRAS = {c: Valoracao.NEUTRA for c in CircunstanciaJudicial}


def com(desfavoraveis):
    valores = dict(TODAS_NEUTRAS)
    for circunstancia in desfavoraveis:
        valores[circunstancia] = Valoracao.DESFAVORAVEL
    return valores


# nenhuma circunstância desfavorável -> pena-base no mínimo da faixa
resultado = calcular_pena_base(faixa_furto, TODAS_NEUTRAS, FracaoDoIntervalo())
assert resultado.pena_base == faixa_furto.minimo
assert resultado.passo.valor_antes == faixa_furto.minimo
assert resultado.passo.valor_depois == faixa_furto.minimo
assert resultado.passo.dispositivo == "CP.art155"

# uma circunstância desfavorável, estratégia 1/8 do intervalo
circunstancias_1 = com([CircunstanciaJudicial.CULPABILIDADE])
resultado_1 = calcular_pena_base(faixa_furto, circunstancias_1, FracaoDoIntervalo())
incremento_esperado = Fracao(1, 8).aplicar(1095)
assert resultado_1.pena_base.dias == faixa_furto.minimo.dias + incremento_esperado

# todas as 8 desfavoráveis nunca ultrapassa o máximo da faixa (mesmo com truncamento por fração)
todas_desfavoraveis = {c: Valoracao.DESFAVORAVEL for c in CircunstanciaJudicial}
resultado_max = calcular_pena_base(faixa_furto, todas_desfavoraveis, FracaoDoIntervalo())
assert resultado_max.pena_base <= faixa_furto.maximo

# com um intervalo múltiplo de 8, 8 circunstâncias desfavoráveis batem exatamente no máximo
faixa_multipla_de_8 = Faixa(minimo=Pena(0), maximo=Pena(800), origem="teste")
resultado_max_exato = calcular_pena_base(faixa_multipla_de_8, todas_desfavoraveis, FracaoDoIntervalo())
assert resultado_max_exato.pena_base == faixa_multipla_de_8.maximo

# circunstância faltando ou desconhecida é rejeitada
try:
    calcular_pena_base(faixa_furto, {CircunstanciaJudicial.CULPABILIDADE: Valoracao.NEUTRA}, FracaoDoIntervalo())
    raise AssertionError("deveria exigir as 8 circunstâncias")
except ValueError:
    pass

print("Fase 1 (pena-base): OK")

Fase 1 (pena-base): OK
